# Large-Scale Portfolio Briefs Generation

## Step 1: Load Company Identifiers from CSV File

This first step reads the CSV file containing company information and extracts all unique company identifiers (RP_ENTITY_ID). These identifiers are used to specify which companies to generate briefs for.

**What it does:**
- Opens the CSV file with company data
- Finds the column containing company IDs
- Removes any empty or duplicate entries
- Creates a clean list of company identifiers for processing

**Output:** A list of unique company IDs that will be used in subsequent steps.


**Note:** All required dependencies (pandas, requests, xlsxwriter, ipython, jupyterlab) should be installed from `requirements.txt` before running this notebook. See the README for installation instructions.


In [ ]:
# Read CSV and produce comma-separated RP_ENTITY_ID string
import pandas as pd
import json

df = pd.read_csv("static/data/US_100.csv", dtype=str)

col = next((c for c in df.columns if c.strip().upper() == "RP_ENTITY_ID"), None)
if col is None:
    raise ValueError("RP_ENTITY_ID column not found in static/data/US_100.csv")

ids = (
    df[col]
    .astype(str)
    .str.strip()
    .replace({"": None})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(len(ids))
#print(json.dumps(ids))
# rp_entity_ids_csv now contains the comma-separated IDs

## Step 2: Define Search Phrases (Topics)

This step defines short phrases used like web-style semantic search queries for each company. They steer which information is retrieved and summarized.

**What it does:**
- Sets up keyword-rich phrases customized with the company name (`{entity}`)
- Covers earnings and metrics, guidance, strategy, contracts, and products

**Note:** Adjust wording to match how your briefing service interprets search-style topics.


In [ ]:
# Build request payload (adjust dates/topics as needed)
TOPICS = [
    # Earnings & Financial Performance
    "{entity} earnings quarterly results revenue profit margins financial performance",
    "{entity} guidance outlook forecast full year operational revision",

    # Strategy & Business Development
    "{entity} strategic initiatives restructuring pivot transformation announcement",
    "{entity} acquisition merger divestiture deal M&A transaction",

    # Leadership & Organization
    "{entity} CEO CFO executive departure appointment leadership change",

    # Commercial & Market Activity
    "{entity} contract wins losses renewals customer deals RFP",
    "{entity} market share competitive position gains losses",
    "{entity} competitive response rival threat market disruption",

    # Product & Innovation
    "{entity} new product launch pipeline R&D development announcement",

    # Operations & Supply Chain
    "{entity} operational disruption capacity constraint production outage",
    "{entity} supply chain shortage logistics input cost pressure",
    "{entity} production milestone efficiency improvement operational performance",

    # Cost Management
    "{entity} cost cutting restructuring layoffs headcount reduction expenses",

    # Regulatory & Legal
    "{entity} regulatory investigation fine penalty compliance enforcement",
    "{entity} litigation lawsuit settlement court ruling legal exposure",

    # Macro & Industry Trends
    "{entity} macroeconomic impact inflation interest rates consumer demand",
    "{entity} industry trend sector disruption structural shift competitive dynamics",

    # Capital Allocation & Financing
    "{entity} capital allocation buyback dividend shareholder returns investment",
    "{entity} dividend increase reduction suspension shareholder return program",
    "{entity} bond issuance debt refinancing credit facility covenant",
    "{entity} credit rating upgrade downgrade outlook watch Moody's S&P Fitch",

    # Market Sentiment & Events
    "{entity} investor sentiment analyst narrative valuation re-rating",
    "{entity} near term catalyst risk event earnings outlook",
    "{entity} unexpected disclosure unusual trading short interest insider",
    "{entity} activist investor stake shareholder engagement governance pressure",
]

## Step 3: Configure Batch Processing and API Settings

This step sets up the configuration for generating briefs, including how many companies to process at once and the date range for the reports.

**What it does:**
- **Batch Size:** Determines how many companies are processed together (20 companies per batch for this example)
- **Company Selection:** Selects the first 100 companies from the list (can be changed to process all companies)
- **Date Range:** Sets the time period for the briefing (start and end dates)
- **API Configuration:** Sets up authentication and the API endpoint URL, if required
- **Processing Options:** Configures how the system prioritizes information (freshness, source ranking, novelty detection)

**Key Settings:**
- `BATCH_SIZE`: Number of companies processed per request (recommended: 50 for production)
- `report_start_date` and `report_end_date`: The time window for gathering information
- `novelty`: Whether to filter for only new or unique information
- `source_rank_boost` and `freshness_boost`: Control how sources are prioritized


In [ ]:
import os
import json
import requests
import pandas as pd

# Batch size, go with 50 for prd use case
BATCH_SIZE = 50

# Take first 100
companies = ids

#for prd use case use 
#companies = ids

print("Using", len(companies), "companies")
#print(json.dumps(companies))

# service uses APIKeyQuery named `token` per your OpenAPI; set env var API_TOKEN (or TOKEN/API_KEY)
token = os.environ.get("API_TOKEN") or os.environ.get("TOKEN") or os.environ.get("API_KEY")
params = {"token": token} if token else {}

payload = {
    "entities": companies,
    "report_start_date": "2025-10-27",
    "report_end_date": "2025-11-03",
    "novelty": True,
    "sources": None,
    "topics": TOPICS,
    "source_rank_boost": 10,
    "freshness_boost": 8,
    "disable_introduction": True,

}

# API call settings
# Update this to the actual create endpoint
API_URL = "http://localhost:8000/briefs/create"



## Step 4: Set Output Folder and File Names

All generated artifacts go under the **`output/`** directory (created if missing):
- **Request summary JSON:** batch request metadata (status, timestamps, entity counts)
- **Combined report JSON:** full briefing data and `source_metadata`
- **Excel export** (Step 10): uses the same folder via `OUT_XLSX`

Downstream cells read `BRIEF_REPORT_FILE` from this path after a successful batch run.


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRIEF_SUMMARY_FILE = OUTPUT_DIR / "briefs_request2_summaries1000.json"
BRIEF_REPORT_FILE = OUTPUT_DIR / "combined_briefs2_report1000.json"
OUT_XLSX = OUTPUT_DIR / "entities_bullets_1000.xlsx"

## Step 5: Process Companies in Batches and Generate Briefs

This is the main processing step that generates briefing reports for all companies. It works by sending requests to the API in batches and waiting for each batch to complete before moving to the next.

**What it does:**
1. **Splits companies into batches** to avoid overwhelming the API
2. **Submits each batch** to the briefing service
3. **Monitors progress** by checking the status of each request
4. **Waits for completion** (up to 10 minutes per batch)
5. **Collects results** from all batches into a single combined report
6. **Saves the results** under `output/` (see Step 4) as JSON files

**Key Features:**
- **Error Handling:** If a batch fails, it records the error and continues with the next batch
- **Status Polling:** Checks every 10 seconds to see if a batch is complete
- **Automatic Merging:** Combines all batch results into one unified report
- **Progress Tracking:** Shows which batch is being processed and when it completes

**Output:** Two JSON files in `output/`: combined report and request summaries.


In [ ]:
import time
import copy
import json
import traceback
import requests
from datetime import datetime 

# Get the current date and time
current_datetime = datetime.now()

# Print the full date and time
print(f"Batch Starting date and time: {current_datetime}")

def _status_url_for(request_id: str) -> str:
    # Update this to the actual status endpoint
    status_url = f"http://localhost:8000/briefs/status/{request_id}"
    return status_url

combined_entity_reports = []
combined_source_metadata = {}
request_summaries = {}

for start in range(0, len(companies), BATCH_SIZE):
    batch = companies[start:start + BATCH_SIZE]
    payload_batch = copy.deepcopy(payload)
    payload_batch["companies"] = batch

    try:
        print(f"Submitting batch {start + 1}-{start + len(batch)} ({len(batch)} entities)...")
        resp = requests.post(API_URL, params=params, json=payload_batch, timeout=180)
        resp.raise_for_status()
        create_resp = resp.json()
    except Exception as e:
        print("Create request failed for batch starting at", start, ":", e)
        traceback.print_exc()
        # store failure summary with no request id
        request_summaries[f"batch_{start}"] = {
            "start_date": payload_batch.get("report_start_date"),
            "end_date": payload_batch.get("report_end_date"),
            "logs": getattr(e, "args", str(e)),
            "report_title": None,
            "watchlist_id": None,
            "status": "create_failed",
            "rp_entity_ids": list(batch),
        }
        continue

    request_id = create_resp.get("request_id")
    # capture immediate create-level logs/title if present
    immediate_report = create_resp.get("report", {}) or {}
    request_summaries[request_id or f"batch_{start}"] = {
        "start_date": payload_batch.get("report_start_date"),
        "end_date": payload_batch.get("report_end_date"),
        "logs": create_resp.get("logs") or immediate_report.get("logs"),
        "report_title": immediate_report.get("report_title") or create_resp.get("report_title"),
        "watchlist_id": immediate_report.get("watchlist_id") or create_resp.get("watchlist_id"),
        "status": "submitted",
        "rp_entity_ids": list(batch),
    }

    # If no request_id, maybe synchronous response contained the report already
    if not request_id and immediate_report:
        ers = immediate_report.get("entity_reports", []) or []
        sm = immediate_report.get("source_metadata", {}) or {}
        combined_entity_reports.extend(ers)
        combined_source_metadata.update(sm)
        request_summaries[request_id or f"batch_{start}"]["status"] = "completed_sync"
        print(f"Batch {start}-{start+len(batch)} returned sync report with {len(ers)} entities.")
        continue

    # Poll status until complete/failed or timeout
    status_url = _status_url_for(request_id)
    timeout_seconds = 600  # total wait per batch
    poll_interval = 10
    waited = 0
    final_status_resp = None
    while waited < timeout_seconds:
        try:
            status_resp = requests.get(status_url, params=params, timeout=60)
            status_resp.raise_for_status()
            sjson = status_resp.json()
            status = sjson.get("status") or sjson.get("state") or ""
            if status and status.lower() in ("completed", "done", "success"):
                final_status_resp = sjson
                request_summaries[request_id]["status"] = "completed"
                break
            if status and status.lower() in ("failed", "error"):
                final_status_resp = sjson
                request_summaries[request_id]["status"] = "failed"
                break
            # otherwise still processing
        except Exception as e:
            print("Status check error:", e)
        time.sleep(poll_interval)
        waited += poll_interval

    if not final_status_resp:
        print(f"Timeout waiting for request {request_id}; proceeding to next batch.")
        request_summaries[request_id]["status"] = "timeout"
        continue

    # extract report data if present
    report = final_status_resp.get("report", {}) or final_status_resp
    entity_reports_chunk = report.get("entity_reports", []) or []
    source_meta_chunk = report.get("source_metadata", {}) or {}

    # merge
    combined_entity_reports.extend(entity_reports_chunk)
    # prefer existing keys (do not overwrite) to preserve first-seen metadata
    for k, v in source_meta_chunk.items():
        if k not in combined_source_metadata:
            combined_source_metadata[k] = v

    # update summary fields with final report metadata
    request_summaries[request_id].update({
        "logs": final_status_resp.get("logs") or request_summaries[request_id].get("logs"),
        "report_title": report.get("report_title") or request_summaries[request_id].get("report_title"),
        "watchlist_id": report.get("watchlist_id") or request_summaries[request_id].get("watchlist_id"),
        "entity_count": len(entity_reports_chunk),
        "completed_at": final_status_resp.get("completed_at") or final_status_resp.get("ts")
    })

    print(f"Batch {start + 1}-{start + len(batch)} completed: {len(entity_reports_chunk)} entities added.")

# Ensure every entity report includes rp_entity_id (API may only send entity_id / entity_info.id)
for er in combined_entity_reports:
    ei = er.get("entity_info") or {}
    rid = er.get("rp_entity_id") or er.get("entity_id") or ei.get("id")
    if rid is not None and str(rid).strip():
        er["rp_entity_id"] = str(rid).strip()

# final combined report
combined_report = {
    "entity_reports": combined_entity_reports,
    "source_metadata": combined_source_metadata
}

# persist results
with open(BRIEF_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(combined_report, f, ensure_ascii=False, indent=2)

with open(BRIEF_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(request_summaries, f, ensure_ascii=False, indent=2)

print(f"Accumulated {len(combined_entity_reports)} entity_reports and {len(combined_source_metadata)} source_metadata entries across {len(request_summaries)} requests.")

# Get the current date and time
current_datetime = datetime.now()

# Print the full date and time
print(f"Batch Completion date and time: {current_datetime}")

## At this stage we should have briefing of selected companies

## Step 7: Set Up Display Functions for Notebook Viewing (Reference Purpose Only)

This step defines helper functions that format and display the briefing reports in a readable way within the Jupyter notebook. These functions are used later to show the results.

**What it does:**
- **`render_source_reference()`:** Formats source information (news articles, reports) with links and metadata
- **`present_entity_report()`:** Creates a nicely formatted display of a company's briefing report with:
  - Company name, sector, industry, and country
  - Numbered bullet points summarizing key information
  - Source links for each bullet point
  - Summary statistics

**Note:** These functions are defined here but used in the next step to display results.


In [ ]:
# For Presentation on Notebook 
from IPython.display import display, Markdown, HTML
import pandas as pd
import json
from pathlib import Path
from pprint import pprint
from typing import Tuple, Dict, Any, Optional
import html


def render_source_reference(
    source_id: str,
    source_metadata: Optional[Dict[str, Any]],
    show_highlights: bool = True,
    snippet_length: int = 300
) -> Tuple[str, Dict[str, Any]]:
    """
    Return (markdown_str, metadata_dict) for a given source id using the provided source_metadata map.
    - markdown_str: ready to display in a Jupyter cell via display(Markdown(...))
    - metadata_dict: the raw meta dict (for programmatic use)
    Behavior follows the selected lines: uses source_name/headline/url from the meta if present.
    """
    def _truncate(s: Optional[str], n: int) -> str:
        if not s:
            return ""
        return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + "…"

    if not source_metadata:
        md = f"`{source_id}` — (no source_metadata provided)"
        return md, {}

    meta = source_metadata.get(source_id) or {}
    if not meta:
        md = f"`{source_id}` — (not found in source_metadata)"
        return md, {}

    # chosen display fields (mirrors your selected snippet)
    name_or_headline = meta.get("source_name") or meta.get("headline") or source_id
    url = meta.get("url")
    ts = meta.get("ts")
    source_key = meta.get("source_key")
    text = meta.get("text")
    highlights = meta.get("highlights", [])

    # escape to avoid accidental HTML injection when rendering
    safe_name = html.escape(name_or_headline)
    safe_ts = html.escape(str(ts)) if ts else ""
    safe_key = html.escape(str(source_key)) if source_key else ""
    safe_snippet = html.escape(_truncate(text, snippet_length))

    # build markdown
    link_part = f"[{safe_name}]({html.escape(url)})" if url else f"**{safe_name}**"
    meta_parts = []
    if safe_ts:
        meta_parts.append(f"`{safe_ts}`")
    if safe_key:
        meta_parts.append(f"`{safe_key}`")
    meta_line = " • ".join(meta_parts)
    md_lines = [f"{link_part}  \n{meta_line}" if meta_line else f"{link_part}"]

    if safe_snippet:
        md_lines.append(f"\n> {safe_snippet}\n")

    if show_highlights and highlights:
        # highlights expected as list of {pnum:int, snum:int} or plain strings
        hl_lines = []
        for h in highlights[:6]:  # limit shown highlights
            if isinstance(h, dict):
                hl_lines.append(f"- paragraph {h.get('pnum')}, sentence {h.get('snum')}")
            else:
                hl_lines.append(f"- {html.escape(str(h))}")
        md_lines.append("**Highlights:**\n" + "\n".join(hl_lines))

    markdown = "\n\n".join(md_lines)
    return markdown, meta, link_part

def present_entity_report(entity: dict, source_metadata: dict | None = None, top_n: int | None = None):
    """
    Nicely render a single entity report for a financial analyst in a Jupyter notebook.
    - entity: the JSON object (keys: entity_id / rp_entity_id, entity_info, content)
    - source_metadata: optional dict mapping source_id -> metadata (url, source_name, headline)
    - top_n: limit number of bullet points shown
    """
    ei = entity.get("entity_info", {})
    name = ei.get("name", "Unknown")
    rp_entity_id = entity.get("rp_entity_id") or entity.get("entity_id") or ei.get("id") or "N/A"
    ticker = ei.get("ticker", "")
    sector = ei.get("sector", "—")
    industry = ei.get("industry", "—")
    country = ei.get("country", "—")
    webpage = ei.get("webpage")

    header = (
        f"## {name}\n"
        f"**RP_ENTITY_ID:** `{rp_entity_id}`  •  **Sector:** {sector}  •  **Industry:** {industry}  •  **Country:** {country}\n"
    )
    if webpage:
        header += f"[Website]({webpage})\n"
    display(Markdown(header))

    bullets = entity.get("content", []) or []
    if not bullets:
        display(Markdown("_No bullet points found for this entity._"))
        return

    # Summary metrics
    num_bullets = len(bullets)
    # gather source counts
    all_srcs = []
    for b in bullets:
        all_srcs.extend(b.get("sources", []))
    src_counts = pd.Series(all_srcs).value_counts()
    top_sources = src_counts.index.tolist()[:5]
    summary_md = f"**Bullet points:** {num_bullets}  •  **Top sources (ids):** {', '.join(top_sources) if top_sources else 'None'}\n"
    display(Markdown(summary_md))

    # Show bullets (numbered) with source links if metadata provided
    limit = top_n if top_n is not None else num_bullets
    for i, b in enumerate(bullets[:limit], start=1):
        text = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []
        # resolve sources to friendly links/names if metadata available
        resolved = []
        markdowns = []
        for s in srcs:
            if source_metadata and s in source_metadata:
                meta = source_metadata[s]
                name_or_headline = meta.get("source_name") or meta.get("headline") or s
                url = meta.get("url")
                if url:
                    resolved.append(f"[{name_or_headline}]({url})")
                else:
                    resolved.append(f"{name_or_headline} ({s})")
            else:
                resolved.append(s)

            markdown, meta, linkpart = render_source_reference(s, source_metadata)
            markdowns.append(linkpart)

        src_line = ", ".join(resolved) if resolved else "None"
        updated_src_line = ", ".join(markdowns) if markdowns else "None"
        display(Markdown(f"{i}. {text}\n\n**Sources:** {updated_src_line}\n"))

        
    # Provide a small table for quick export / analysis
    df = pd.DataFrame([
        {"rp_entity_id": rp_entity_id, "bullet": b.get("bullet_point", ""), "sources": b.get("sources", [])}
        for b in bullets
    ])
    display(Markdown("**Raw table (for copy/export):**"))
    display(df.head(limit))


## Step 8: Load Saved Briefing Report

This step loads the briefing report that was saved during batch processing. The report contains all company briefings and their source information.

**What it does:**
- Opens the saved JSON file containing the combined briefing report
- Extracts the company reports and source metadata
- Makes the data available for display or export in subsequent steps

**Output:** 
- `entities`: List of all company briefing reports
- `source_metadata`: Dictionary mapping source IDs to source information (URLs, headlines, publication dates)


In [ ]:
#read entities and source_metadata from save file 
import json
from pathlib import Path
from pprint import pprint

REPORT_PATH = Path(BRIEF_REPORT_FILE)

if not REPORT_PATH.exists():
    raise FileNotFoundError(
        f"{REPORT_PATH} not found. Run Step 4 (output paths) then the batching cell that writes {BRIEF_REPORT_FILE.name} first."
    )

with REPORT_PATH.open("r", encoding="utf-8") as f:
    combined = json.load(f)

entities = combined.get("entity_reports", []) or []
source_metadata = combined.get("source_metadata", {}) or {}

print(f"Loaded combined report: {len(entities)} entities, {len(source_metadata)} source_metadata entries.")
# optional quick inspect
if entities:
    print("First entity keys:", list(entities[0].keys()))
if source_metadata:
    print("Sample source id:", next(iter(source_metadata.keys())))

## Step 9: Display Sample Reports in Notebook (Reference Purpose Only)

This step displays a preview of the briefing reports directly in the notebook. It shows the first 5 companies with their top 5 bullet points each.

**What it does:**
- Takes the first 5 companies from the loaded report
- Formats each company's briefing with:
  - Company information (name, sector, industry)
  - Key bullet points summarizing important information
  - Clickable source links for each bullet point
- Displays everything in a clean, readable format

**Purpose:** Allows you to review the briefing reports immediately in the notebook before exporting to other formats.


In [ ]:
# Now we can present the report in a notebook
reportable_entities = entities[:3] # picking first 5 entities for presentation
for ent in reportable_entities:
    present_entity_report(ent, source_metadata=source_metadata, top_n=5)

## Step 10: Export Briefing Report to Excel (Reference Purpose Only)

This final step converts the briefing reports into an Excel spreadsheet format that can be easily shared, analyzed, or imported into other tools.

**What it does:**
- Creates a structured table with one row per bullet point
- Includes company information (name, sector, industry, country, website) on the first row for each company
- Lists bullet points with their associated sources
- Adds clickable hyperlinks to the source URLs in the Excel file
- Saves the workbook under `output/` (same folder as Step 4; see `OUT_XLSX`)

**Output:** An Excel file that can be opened in Microsoft Excel, Google Sheets, or any spreadsheet application. Each row contains a bullet point, and the source column includes clickable links to the original articles or reports.


In [ ]:
# Export report to Excel 

import pandas as pd
import json
from IPython.display import display

# OUT_XLSX is set in Step 4 (under output/)

source_map = source_metadata

rows = []
link_meta = []  # parallel list of lists of urls (or None) for each row

for e1 in entities:
    # print (e1)
    ei = e1.get("entity_info", {}) or {}
    rp_entity_id = e1.get("rp_entity_id") or e1.get("entity_id") or ei.get("id") or ""
    name = ei.get("name") or ei.get("id") or e1.get("entity_id", "")
    sectors = ei.get("sector", "")
    industry = ei.get("industry", "")
    country = ei.get("country", "")
    website = ei.get("webpage", "") or ei.get("web_site", "")

    bullets = e1.get("content", []) or []
    if not bullets:
        rows.append({
            "rp_entity_id": rp_entity_id,
            "entity name": name,
            "sectors": sectors,
            "industry": industry,
            "country": country,
            "website": website,
            "bulletpoint": "",
            "source": ""
        })
        link_meta.append([])  # no links
        continue

    for i, b in enumerate(bullets):
        bp = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []

        resolved_displays = []
        resolved_urls = []
        for s in srcs:
            meta = source_map.get(s) or {}
            headline = meta.get("headline") or meta.get("source_name")
            url = meta.get("url")
            display_text = headline if headline else s
            resolved_displays.append(display_text)
            resolved_urls.append(url)  # may be None

        source_field = "; ".join(resolved_displays)
        first_url = next((u for u in resolved_urls if u), None)  # first available URL (or None)

        if i == 0:
            rows.append({
                "rp_entity_id": rp_entity_id,
                "entity name": name,
                "sectors": sectors,
                "industry": industry,
                "country": country,
                "website": website,
                "bulletpoint": bp,
                "source": source_field
            })
        else:
            rows.append({
                "rp_entity_id": "",
                "entity name": "",
                "sectors": "",
                "industry": "",
                "country": "",
                "website": "",
                "bulletpoint": bp,
                "source": source_field
            })

        link_meta.append([first_url])  # store first url (or [None])

df_out = pd.DataFrame(rows, columns=[
    "rp_entity_id", "entity name", "sectors", "industry", "country", "website", "bulletpoint", "source"
])

# Write to Excel with first source as hyperlink (cell displays all source texts, link opens first URL)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="Briefs")
    workbook = writer.book
    worksheet = writer.sheets["Briefs"]

    # find source column index
    src_col = df_out.columns.get_loc("source")

    # rows in sheet start at 1 (0 is header)
    for r_idx, lm in enumerate(link_meta, start=1):
        first_url = lm[0] if lm else None
        if first_url:
            display_text = df_out.iloc[r_idx - 1]["source"] or first_url
            # write_url will display the provided string but link to first_url
            worksheet.write_url(r_idx, src_col, first_url, string=display_text)

print(f"Written {len(df_out)} rows to {OUT_XLSX}")
display(df_out.head(20))
